In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# .env を読み込む
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "agent-book"

In [3]:
from langchain_openai import OpenAI

model = OpenAI(model="gpt-4o-mini", temperature=0)
output = model.invoke("自己紹介してください。")
print(output)

 こんにちは、私はAIアシスタントです。あなたの質問に答えたり、情報を提供したりするためにここにいます。どんなことでもお手伝いできることがあれば教えてください。 何か特定のことについてお話ししたいですか？ それとも、何か質問がありますか？ どうぞお気軽にお尋ねください。 どんなことでもお手伝いできることがあれば教えてください。 何か特定のことについてお話ししたいですか？ それとも、何か質問がありますか？ どうぞお気軽にお尋ねください。 どんなことでもお手伝いできることがあれば教えてください。 何か特定のことについてお話ししたいですか？ それとも、何か質問がありますか？ どうぞお気軽にお尋ねください。 どんなことでもお手伝いできることがあれば教えてください。 何か特定のことについてお話ししたいですか？ それとも、何か質問がありますか？ どうぞお気軽にお尋ねください。 ど


In [4]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

messages = [
    SystemMessage("You are a helpful assistant."),
    HumanMessage("こんにちは！私は賢二と言います！"),
    AIMessage(content="こんにちは、賢二さん！どのようにお手伝いができますか？"),
    HumanMessage(content="私の名前がわかりますか？"),
]

ai_message = model.invoke(messages)
print(ai_message.content)

はい、賢二さんですね！何か特別なことについてお話ししたいことがありますか？


In [5]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
messages = [
    SystemMessage("You are a helpful assistant."),
    HumanMessage("こんにちは"),
]

for chunk in model.stream(messages):
    print(chunk.content, end="", flush=True)

こんにちは！どのようにお手伝いできますか？

In [6]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("""以下の料理のレシピを考えてください。

料理名：{dish}""")

prompt_value = prompt.invoke({"dish": "たこ焼き"})
print(prompt_value)

text='以下の料理のレシピを考えてください。\n\n料理名：たこ焼き'


In [7]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        MessagesPlaceholder("chat_history", optional=True),
        ("human", "{input}"),
    ]
)

prompt_value = prompt.invoke(
    {
        "chat_history": [
            HumanMessage(content="こんにちは！私は賢二と言います！"),
            AIMessage("こんにちは、賢二さん！どのようにお手伝いできますか？"),
        ],
        "input": "私の名前がわかりますか？",
    }
)
print(prompt_value)

messages=[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='こんにちは！私は賢二と言います！', additional_kwargs={}, response_metadata={}), AIMessage(content='こんにちは、賢二さん！どのようにお手伝いできますか？', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='私の名前がわかりますか？', additional_kwargs={}, response_metadata={})]


In [8]:
from langsmith import Client

client = Client()
prompt = client.pull_prompt("oshima/recipe")

prompt_value = prompt.invoke({"dish": "たこ焼き"})
print(prompt_value)

messages=[SystemMessage(content='ユーザーが入力した料理のレシピを考えてください。', additional_kwargs={}, response_metadata={}), HumanMessage(content='たこ焼き', additional_kwargs={}, response_metadata={})]


In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "user",
            [
                {"type": "text", "text": "画像を説明してください。"},
                {"type": "image_url", "image_url": {"url": "{image_url}"}},
            ],
        ),
    ]
)
image_url = "https://raw.githubusercontent.com/yoshidashingo/langchain-book/main/assets/cover.jpg"

prompt_value = prompt.invoke({"image_url": image_url})

In [10]:
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
ai_message = model.invoke(prompt_value)
print(ai_message.content)

この画像は、書籍の表紙を示しています。タイトルは「ChatGPT/LangChainによるチャットシステム構築【実践】入門」で、著者は吉田真吾と大嶋秀樹です。表紙にはカラフルな鳥のイラストが描かれており、背景は青色です。また、「ChatGPT」という文字が大きく表示されています。全体的に、技術的な内容を扱った書籍であることが伝わるデザインになっています。


In [11]:
from pydantic import BaseModel, Field

class Recipe(BaseModel):
    ingredients: list[str] = Field(description="ingredients of the dish")
    steps: list[str] = Field(description="step to make the dish")

In [12]:
from langchain_core.output_parsers import PydanticOutputParser

output_parser = PydanticOutputParser(pydantic_object=Recipe)

In [13]:
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"ingredients": {"description": "ingredients of the dish", "items": {"type": "string"}, "title": "Ingredients", "type": "array"}, "steps": {"description": "step to make the dish", "items": {"type": "string"}, "title": "Steps", "type": "array"}}, "required": ["ingredients", "steps"]}
```


In [14]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "ユーザーが入力した料理のレシピを考えてください。\n\n"
            "{format_instructions}",
        ),
        ("human", "{dish}"),
    ]
)

prompt_with_format_instructions = prompt.partial(
    format_instructions=format_instructions
)

In [15]:
prompt_value = prompt_with_format_instructions.invoke({"dish": "たこ焼き"})
print("=== role: system ===")
print(prompt_value.messages[0].content)
print("=== role: user ===")
print(prompt_value.messages[1].content)

=== role: system ===
ユーザーが入力した料理のレシピを考えてください。

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"ingredients": {"description": "ingredients of the dish", "items": {"type": "string"}, "title": "Ingredients", "type": "array"}, "steps": {"description": "step to make the dish", "items": {"type": "string"}, "title": "Steps", "type": "array"}}, "required": ["ingredients", "steps"]}
```
=== role: user ===
たこ焼き


In [16]:
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

ai_message = model.invoke(prompt_value)
print(ai_message.content)

{
  "ingredients": [
    "たこ焼き粉 200g",
    "水 600ml",
    "卵 2個",
    "茹でたタコ 200g",
    "ネギ 50g",
    "紅しょうが 30g",
    "天かす 50g",
    "サラダ油 適量",
    "ソース 適量",
    "マヨネーズ 適量",
    "かつお節 適量",
    "青のり 適量"
  ],
  "steps": [
    "ボウルにたこ焼き粉、水、卵を入れてよく混ぜる。",
    "ネギ、紅しょうが、茹でたタコを小さく切り、混ぜた生地に加える。",
    "たこ焼き器を熱し、型にサラダ油を塗る。",
    "生地を型の8分目まで流し込み、天かすを加える。",
    "生地が固まり始めたら、竹串を使ってひっくり返す。",
    "全体がこんがりと焼き色がつくまで、何度かひっくり返しながら焼く。",
    "焼き上がったたこ焼きを皿に盛り、ソース、マヨネーズ、かつお節、青のりをトッピングして完成。"
  ]
}


In [17]:
recipe = output_parser.invoke(ai_message)
print(type(recipe))
print(recipe)

<class '__main__.Recipe'>
ingredients=['たこ焼き粉 200g', '水 600ml', '卵 2個', '茹でたタコ 200g', 'ネギ 50g', '紅しょうが 30g', '天かす 50g', 'サラダ油 適量', 'ソース 適量', 'マヨネーズ 適量', 'かつお節 適量', '青のり 適量'] steps=['ボウルにたこ焼き粉、水、卵を入れてよく混ぜる。', 'ネギ、紅しょうが、茹でたタコを小さく切り、混ぜた生地に加える。', 'たこ焼き器を熱し、型にサラダ油を塗る。', '生地を型の8分目まで流し込み、天かすを加える。', '生地が固まり始めたら、竹串を使ってひっくり返す。', '全体がこんがりと焼き色がつくまで、何度かひっくり返しながら焼く。', '焼き上がったたこ焼きを皿に盛り、ソース、マヨネーズ、かつお節、青のりをトッピングして完成。']


In [18]:
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

ai_message = AIMessage(content="こんにちは。私はAIアシスタントです。")
output = output_parser.invoke(ai_message)
print(type(output))
print(output)

<class 'langchain_core.messages.base.TextAccessor'>
こんにちは。私はAIアシスタントです。


In [19]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "ユーザーが入力した料理のレシピを考えてください。"),
        ("human", "{dish}"),
    ]
)

model = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

In [20]:
chain = prompt | model

In [21]:
ai_message = chain.invoke({"dish": "たこ焼き"})
print(ai_message.content)

たこ焼きのレシピをご紹介します！外はカリッと、中はトロッとした美味しいたこ焼きを作りましょう。

### 材料（約20個分）
- たこ焼き粉：200g
- 水：600ml
- 卵：1個
- ゆでだこ：150g（小さく切る）
- 青ねぎ：適量（小口切り）
- 天かす：適量
- 紅しょうが：適量（みじん切り）
- サラダ油：適量
- たこ焼きソース：適量
- マヨネーズ：適量
- 青のり：適量
- かつお節：適量

### 作り方
1. **生地を作る**  
   ボウルにたこ焼き粉、水、卵を入れ、よく混ぜ合わせて生地を作ります。ダマがなくなるまでしっかり混ぜましょう。

2. **具材の準備**  
   ゆでだこは小さく切り、青ねぎ、天かす、紅しょうがもそれぞれ準備します。

3. **たこ焼き器の準備**  
   たこ焼き器を中火で熱し、各穴にサラダ油をたっぷりと塗ります。

4. **生地を流し込む**  
   熱したたこ焼き器の穴に生地を流し込み、穴の8分目まで入れます。

5. **具材を入れる**  
   生地の上に、切ったたこ、青ねぎ、天かす、紅しょうがをそれぞれの穴に入れます。

6. **生地を追加**  
   具材の上からさらに生地を流し込み、穴が満たされるようにします。

7. **焼く**  
   たこ焼きが焼けてきたら、周りが固まってきたら、竹串や専用のピックを使って、たこ焼きをひっくり返します。全体が均一に焼けるように、何度も回転させます。

8. **焼き上がり**  
   たこ焼きがこんがりと焼き色がついたら、取り出して皿に盛ります。

9. **トッピング**  
   たこ焼きソースをたっぷりかけ、マヨネーズをかけ、青のりとかつお節を振りかけて完成です！

### おすすめの食べ方
熱々のたこ焼きをそのまま食べるのも良いですが、好みでポン酢や辛子を添えても美味しいです。お好みで具材を変えて、自分だけのオリジナルたこ焼きを楽しんでください！


In [22]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | model | StrOutputParser()
output = chain.invoke({"dish": "たこ焼き"})
print(output)

たこ焼きのレシピをご紹介します！外はカリッと、中はトロッとした美味しいたこ焼きを作りましょう。

### 材料（約20個分）
- たこ焼き粉：200g
- 水：600ml
- 卵：1個
- ゆでたこ：100g（小さく切る）
- 青ねぎ：適量（小口切り）
- 天かす：適量
- 紅しょうが：適量（みじん切り）
- サラダ油：適量
- たこ焼きソース：適量
- マヨネーズ：適量
- 青のり：適量
- かつお節：適量

### 作り方
1. **生地を作る**  
   ボウルにたこ焼き粉、水、卵を入れ、よく混ぜて滑らかな生地を作ります。

2. **具材の準備**  
   ゆでたこは小さく切り、青ねぎ、天かす、紅しょうがもそれぞれ準備します。

3. **たこ焼き器の準備**  
   たこ焼き器を中火で熱し、各穴にサラダ油をたっぷりと塗ります。

4. **生地を流し込む**  
   熱したたこ焼き器の穴に生地を流し込み、穴の8分目まで入れます。

5. **具材を入れる**  
   生地の上に、切ったたこ、青ねぎ、天かす、紅しょうがをそれぞれの穴に入れます。

6. **焼く**  
   生地が少し固まってきたら、竹串や専用の道具を使って、たこ焼きをひっくり返します。全体が均一に焼けるように、何度も回転させながら焼きます。

7. **焼き上がり**  
   きれいな焼き色がついたら、たこ焼きを取り出します。

8. **盛り付け**  
   皿に盛り付け、たこ焼きソース、マヨネーズ、青のり、かつお節をトッピングして完成です！

### お好みで
- 具材はお好みでアレンジできます。チーズやエビ、ウィンナーなども美味しいです。
- 辛いものが好きな方は、辛子や一味唐辛子をトッピングしても良いでしょう。

ぜひ、お家でたこ焼きを楽しんでください！


In [23]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class Recipe(BaseModel):
    ingredients: list[str] = Field(description="ingredients of the dish")
    steps: list[str] = Field(description="steps to make the dish")

output_parser = PydanticOutputParser(pydantic_object=Recipe)

In [24]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "ユーザーが入力した料理のレシピを考えてください。\n\n{format_instructions}"),
        ("human", "{dish}"),
    ]
)

prompt_with_format_instructions = prompt.partial(
    format_instructions=output_parser.get_format_instructions()
)

model = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind(
    response_format={"type": "json_object"}
)

In [25]:
chain = prompt_with_format_instructions | model | output_parser

In [26]:
recipe = chain.invoke({"dish": "茶碗蒸し"})
print(type(recipe))
print(recipe)

<class '__main__.Recipe'>
ingredients=['卵 3個', 'だし 400ml', '醤油 大さじ1', 'みりん 大さじ1', '塩 少々', '鶏肉 50g (一口大に切る)', '椎茸 2枚 (薄切り)', '銀杏 4粒', '青ねぎ 適量 (小口切り)'] steps=['卵をボウルに割り入れ、よくかき混ぜる。', 'だし、醤油、みりん、塩を加え、さらに混ぜる。', '鶏肉、椎茸、銀杏を器に入れ、その上から卵液を注ぐ。', '器にラップをし、蒸し器で中火で約15分蒸す。', '蒸し上がったら、青ねぎを散らして完成。']


In [27]:
from langchain_community.document_loaders import GitLoader

def file_filter(file_path: str) -> bool:
    return file_path.endswith((".md", ".mdx"))

loader = GitLoader(
    clone_url="https://github.com/langchain-ai/langchain",
    repo_path="./langchain",
    branch="master",
    file_filter=file_filter,
)

raw_docs = loader.load()

print("loaded:", len(raw_docs))
if raw_docs:
    print(raw_docs[0].metadata)
    print(raw_docs[0].page_content[:300])

loaded: 28
{'source': 'AGENTS.md', 'file_path': 'AGENTS.md', 'file_name': 'AGENTS.md', 'file_type': '.md'}
# Global development guidelines for the LangChain monorepo

This document provides context to understand the LangChain Python project and assist with development.

## Project architecture and context

### Monorepo structure

This is a Python monorepo with multiple independently versioned packages th


In [28]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

docs = text_splitter.split_documents(raw_docs)
print(len(docs))

Created a chunk of size 1455, which is longer than the specified 1000


93


In [29]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [30]:
query = "AWSのS3データを読み込むためのDocument loaderはありますか？"

vector = embeddings.embed_query(query)
print(len(vector))
print(vector)

1536
[0.0203857421875, -0.01128387451171875, 0.031707763671875, -0.0312347412109375, 0.042205810546875, 0.0256805419921875, -0.01568603515625, 0.0195159912109375, 0.0284881591796875, -0.022918701171875, 0.00783538818359375, -0.00785064697265625, -0.0210418701171875, -0.00672149658203125, -0.0022602081298828125, 0.06427001953125, 0.03375244140625, -0.001705169677734375, -0.051177978515625, 0.0248260498046875, 0.0274200439453125, 0.031097412109375, -0.04046630859375, 0.0263519287109375, 0.0213623046875, -0.0213470458984375, -0.0273590087890625, 0.030548095703125, -0.0036449432373046875, -0.102294921875, -0.00890350341796875, -0.05712890625, -0.03515625, 0.046722412109375, -0.023834228515625, 0.0301971435546875, 0.0216217041015625, 0.00945281982421875, -0.0034961700439453125, -0.0386962890625, -0.0010175704956054688, -0.0238494873046875, 0.0216827392578125, 0.0122528076171875, 0.0103607177734375, 0.00319671630859375, 0.01206207275390625, -0.00257110595703125, -0.04541015625, 0.00490570068

In [31]:
print(type(docs))
print(type(docs[0]))
print(docs[0])

<class 'list'>
<class 'langchain_core.documents.base.Document'>
page_content='# Global development guidelines for the LangChain monorepo

This document provides context to understand the LangChain Python project and assist with development.

## Project architecture and context

### Monorepo structure

This is a Python monorepo with multiple independently versioned packages that use `uv`.' metadata={'source': 'AGENTS.md', 'file_path': 'AGENTS.md', 'file_name': 'AGENTS.md', 'file_type': '.md'}


In [32]:
from langchain_chroma import Chroma

db = Chroma(
    collection_name="sample",
    embedding_function=embeddings
)

db.add_documents(docs)

['d9e95fc3-f587-43ec-8f16-0e8f898a6df9',
 '651b787d-01d2-4c1e-8c07-a382c6fbad6c',
 '4b1362d4-ebbc-406c-8acd-b41694d656b4',
 '9768bddc-0a8b-4339-bcd8-0af9ef9e8e77',
 '1192f666-3a92-44c8-b3fe-4e1fb5599e1d',
 '5786ce80-bb52-4554-a2ca-4e7ca2611fcf',
 'ae605d13-c42e-4a0c-957f-eb3d27b54be0',
 '6a71aa11-288a-4036-a416-ec15948ce9f6',
 '5d26fbce-330a-464f-b516-897fd3c87033',
 '80795d44-cd96-4d9e-a0ec-2d4a72aa3f26',
 '3eadfe67-a3b4-417e-a941-3bd9b998beff',
 '4826cffd-c599-4f6f-8ca8-811ec2361734',
 '0f83b583-c9ca-4626-ac70-786a6e4b9b42',
 '2030fb50-9eee-4402-8b62-f094c166498b',
 '7cb604e0-9227-4893-84d2-fc227fcc4020',
 '59f6fbd7-4d5e-41e1-b753-e8abbfc7eb57',
 '2406c66f-2cbd-495e-b87f-f3ed915f5e2a',
 'de155189-7f5e-4c50-9b06-9a43d89a68e9',
 '822f49cd-8139-4273-9fcf-db630d254c15',
 '2a780d10-fd65-40c0-9658-905bb1fe33c2',
 'e4cd72aa-4316-44d0-934e-1a411b191b48',
 'f26af229-4001-45b1-b747-26fed0650174',
 'ca71874a-8653-4b43-b8f6-978bd7e500bb',
 '4c54e620-46a0-4bb1-a294-b84efb5269e6',
 'c09bbcdd-41b6-

In [33]:
retriever = db.as_retriever()

In [35]:
query = "AWSのS3からデータを読み込むためのDocument loaderはありますか？"

context_docs = retriever.invoke(query)
print(f"len = {len(context_docs)}")

first_doc = context_docs[0]
print(f"metadata = {first_doc.metadata}")
print(first_doc.page_content)

len = 4
metadata = {'file_path': 'libs/text-splitters/README.md', 'file_type': '.md', 'file_name': 'README.md', 'source': 'libs/text-splitters/README.md'}
# 🦜✂️ LangChain Text Splitters

[![PyPI - Version](https://img.shields.io/pypi/v/langchain-text-splitters?label=%20)](https://pypi.org/project/langchain-text-splitters/#history)
[![PyPI - License](https://img.shields.io/pypi/l/langchain-text-splitters)](https://opensource.org/licenses/MIT)
[![PyPI - Downloads](https://img.shields.io/pepy/dt/langchain-text-splitters)](https://pypistats.org/packages/langchain-text-splitters)
[![Twitter](https://img.shields.io/twitter/url/https/twitter.com/langchain.svg?style=social&label=Follow%20%40LangChain)](https://x.com/langchain)

Looking for the JS/TS version? Check out [LangChain.js](https://github.com/langchain-ai/langchainjs).

## Quick Install

```bash
pip install langchain-text-splitters
```

## 🤔 What is this?

LangChain Text Splitters contains utilities for splitting into chunks a wide va

In [37]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_template('''\
以下の文脈だけを踏まえて質問に回答してください。

文脈："""
{context}
"""

質問：{question}
''')

model = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

In [38]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

output = chain.invoke(query)
print(output)

文脈にはAWSのS3からデータを読み込むためのDocument loaderに関する情報は含まれていません。したがって、具体的な回答はできません。AWS S3に関連するDocument loaderについての情報は、別の資料やドキュメントを参照する必要があります。
